In [ ]:
import numpy as np
import wfdb
import pywt

import cv2
from scipy import signal

import os
import gc

In [ ]:
def windowed_data(data, wind_len=2500, num_leads=8):
    all_data = []
    for i in range(round(len(data) / wind_len)):
        cut_data = data[i*wind_len:i*wind_len + wind_len]
        if len(cut_data) < wind_len:
            cut_data = data[-wind_len:]
        all_data.append(cut_data)
    return np.array(all_data)[:,:,:8]

In [ ]:
def get_dwt_scalogram(window, wavelet='db4', level=5, shape=(125, 120)):
    h, w = shape
    ecg_data = window.T
    all_scalograms = []
    for lead_signal in ecg_data:

        coeffs = pywt.wavedec(lead_signal, wavelet, level=level)
        stack = [cv2.resize(np.abs(c).reshape(1, -1), (w, 1))[0] for c in coeffs[::-1]]
        scalogram = np.vstack(stack)
        if scalogram.shape[0] != h:
            scalogram = cv2.resize(scalogram, (w, h), interpolation=cv2.INTER_LINEAR)

        all_scalograms.append(np.log1p(scalogram))
    multi_channel_input = np.stack(all_scalograms, axis=-1)
    return multi_channel_input

def prepare_stft_data(window, fs=500, nperseg=256, noverlap=128):
    ecg_data = window.T
    all_spectrograms = []
    for lead_signal in ecg_data:
        
        f, t, Zxx = signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
        amplitude = np.abs(Zxx)
        
        spec_log = np.log10(amplitude + 1e-6)
        spec_norm = (spec_log - np.mean(spec_log)) / (np.std(spec_log) + 1e-8)
        all_spectrograms.append(spec_norm)
    multi_channel_input = np.stack(all_spectrograms, axis=-1)
    
    return multi_channel_input

Данные из Physionet Challenge 2020

In [ ]:
path_full = "cpsc_2018_extra\\"
g_list = ["g1\\","g2\\","g3\\","g4\\"]
RECORDS = []
for g in g_list:
    with open (path_full+g+"RECORDS", 'r', encoding='utf-8') as f:
        RECORDS += f.read().split("\n")[:-1]

In [ ]:
def onehot_group(target):
    oh = np.zeros(12)
    st_t_codes = {'428750005', '164930006', '164934002', '429622005'} # Изменения ST/T
    rare_imp = {'111975006', '164909002', '195042002', '54329005'} # QT, БЛНПГ, и т.д.

    for code in target:
        # 0. Норма
        if code == '427084000' or code == '164865005': 
            oh[0] = 1
        # 1. Изменения ST-T
        elif code in st_t_codes:
            oh[1] = 1
        # 2. Мерцалка/Трепетание (AF/AFL)
        elif code == '164889003' or code == '164890007':
            oh[2] = 1
        # 3. Брадикардия
        elif code == '164861001':
            oh[3] = 1
        # 4. Желудочковая экстрасистолия (PVC)
        elif code == '427172004':
            oh[4] = 1
        # 5. Предсердная экстрасистолия (PAC)
        elif code == '284470004':
            oh[5] = 1
        # 6. Блокада левой ножки (LBBB)
        elif code == '164909002':
            oh[6] = 1
        # 7. Блокада правой ножки (RBBB)
        elif code == '164867002' or code == '164931005':
            oh[7] = 1
        # 8. Отклонение оси влево (LAD)
        elif code == '164873001':
            oh[8] = 1
        # 9. АВ-блокада 1 степени
        elif code == '270492004':
            oh[9] = 1
        # 10. Тахикардии
        elif code == '426177001' or code == '426761007':
            oh[10] = 1
        # 11. Rare Important (Опасные/Важные редкие)
        elif code in rare_imp:
            oh[11] = 1
            
    return oh

In [ ]:
all_code_freqs_gr = np.zeros(12)
for g in g_list:
    for rec_id in RECORDS:
        try:
            rec = wfdb.rdrecord(path_full+g+rec_id)
        except FileNotFoundError:
            continue
        header_rec = rec.comments
        diag = header_rec[2][4:].split(",")
    
        all_code_freqs_gr += onehot_group(diag)
all_code_freqs_gr = all_code_freqs_gr.astype(int)

total_samples = len(RECORDS)
weights = (total_samples - all_code_freqs_gr) / (all_code_freqs_gr + 1e-6)
weights = np.log1p(weights)
weights

In [4]:
num_classes = 12
batch_size_records = 150 # Сохраняем каждые 150 пациентов
window_size = 2500
fs = 500
X_dwt_full, X_stft_full, y = [], [], []

os.makedirs('/processed_data', exist_ok=True)
os.makedirs('/processed_data/train', exist_ok=True)
os.makedirs('/processed_data/val', exist_ok=True)

In [ ]:
def normalize_signal(signal):
    mean = np.mean(signal)
    std = np.std(signal) + 1e-6
    return (signal - mean) / std

In [ ]:
batch_count = 0

for g in g_list:
    for i, rec_id in enumerate(RECORDS):
        try:
            rec = wfdb.rdrecord(path_full+g+rec_id)
        except FileNotFoundError:
            continue
        
        data_rec = rec.p_signal
        
        header_rec = rec.comments
        diag = header_rec[2][4:].split(",")
        
        cut_data = windowed_data(data_rec, window_size)
        for window in cut_data:
            target = onehot_group(diag)
            
            if np.any(target):# непустые векторы
                window_scaled = normalize_signal(window)
                X_dwt_full.append(get_dwt_scalogram(window_scaled))
                X_stft_full.append(prepare_stft_data(window_scaled))
                y.append(onehot_group(diag))
            else:
                continue
        
        # Если накопили достаточно или это последний файл
        if (i + 1) % batch_size_records == 0:
            sub_folder = 'train' if rec_id in train_names else 'val'
            # Превращаем в массивы и сохраняем
            np.save(f'/processed_data/{sub_folder}/X_dwt_{batch_count}.npy', np.array(X_dwt_full, dtype=np.float32))
            np.save(f'/processed_data/{sub_folder}/X_stft_{batch_count}.npy', np.array(X_stft_full, dtype=np.float32))
            np.save(f'/processed_data/{sub_folder}/y_{batch_count}.npy', np.array(y, dtype=np.float32))
            
            # ОЧИСТКА
            X_dwt_full, X_stft_full, y = [], [], []
            gc.collect() # Принудительно очищаем память
            batch_count += 1
            print(f"Batch {batch_count} saved.")
